# Nutrition5k: resultados generales

Comparaciones entre familias. Primero la elección de la cabeza (etapa 1, backbone congelado, los 12 backbones);
después, los finalistas y los cambios de receta (se entrenan con `notebooks/finalists/experiments.ipynb`). Los detalles de cada familia están en `notebooks/<familia>/results.ipynb`.

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from nutrition5k.results import depth_steps, head_sizes, head_steps, load_heads, load_runs, stage2_runs

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

heads = load_heads(split="full")
print(f"{len(heads)} heads of {heads['backbone'].nunique()} backbones")

## Etapa 1: elección de la cabeza

Cada columna es el cambio del PMAE medio de validación al pasar de una cabeza a otra, medio sobre los backbones de
cada familia (`all backbones`: los 12). **Negativo = la segunda cabeza es mejor.** `seed SD` es la SD media entre las
semillas 0, 1 y 2 de la cabeza por defecto (`512 · single 256`): el ruido con el que comparar. Semilla 0, flip.

Nombre de cada cabeza: `trunk · cabeza`; `linear` = una capa lineal a los 5 objetivos, `single 256` = una capa
oculta de 256 compartida, `grouped 256` = una capa oculta de 256 por grupo de objetivos.

### 1. ¿Hace falta la capa oculta de 256?

In [ ]:
head_steps(heads, {
    "0: linear → single 256": ("0 · linear", "0 · single 256"),
    "512: linear → single 256": ("512 · linear", "512 · single 256"),
    "1024x1024: linear → single 256": ("1024x1024 · linear", "1024x1024 · single 256"),
})

### 2. Tamaño del trunk (512 → 1024x1024) y agrupación (single → grouped)

In [ ]:
head_steps(heads, {
    "512 → 1024x1024 (single 256)": ("512 · single 256", "1024x1024 · single 256"),
    "512 → 1024x1024 (grouped 256)": ("512 · grouped 256", "1024x1024 · grouped 256"),
    "single → grouped (512)": ("512 · single 256", "512 · grouped 256"),
    "single → grouped (1024x1024)": ("1024x1024 · single 256", "1024x1024 · grouped 256"),
    "512 · single → 1024x1024 · grouped": ("512 · single 256", "1024x1024 · grouped 256"),
})

### 3. Tamaño de la cabeza frente a la mejora

Media por familia: parámetros de la cabeza (miles), `head_pct` = % de los parámetros del modelo entero (backbone +
cabeza) que son de la cabeza, PMAE de validación, y `params ×` y `Δ val PMAE` respecto a `512 · single 256`.

In [ ]:
head_sizes(heads, ["512 · single 256", "512 · grouped 256", "1024x1024 · single 256", "1024x1024 · grouped 256"])

### Análisis (runs del stage 1 en `full`, 2026-10-31)

- **Capa oculta de 256: sí.** Sin trunk, pasar de lineal a `single 256` baja 5.2 puntos de media (3.6 a 7.7 por
  familia); con trunk 512, 1.0 (0.7 a 1.6). Con 1024x1024 ya casi no aporta (0.2), porque ese trunk ya da la
  no linealidad. Lo que hace falta es al menos una capa no lineal entre las features y la salida.
- **512 → 1024x1024:** −0.20 de media con `single 256` y −0.45 con `grouped 256` (de +0.07 a −0.57 por familia).
  Está por debajo del umbral de empate de 1 punto, pero en algunas familias supera la SD entre semillas (0.19 a
  0.46): puede ser una mejora real, pero pequeña.
- **Cabeza grande entera** (`512 · single 256` → `1024x1024 · grouped 256`): −0.30 de media (de +0.13 a −0.58 por
  familia).
- **single → grouped:** +0.15 con trunk 512 y −0.10 con 1024x1024, con signos distintos según la familia: sin efecto.
- **Tamaño:** `1024x1024 · single 256` tiene de 3.3 a 5.5 veces los parámetros de `512 · single 256`, y
  `1024x1024 · grouped 256` de 4 a 7.3 veces. En MobileNet la cabeza grande es casi la mitad del modelo (46 %, frente
  al 19 %) a cambio de ~0.5 puntos.
- Todo esto es con el backbone congelado, donde la cabeza es lo único que se adapta. Con fine-tuning la diferencia
  podría reducirse (hipótesis, sin medir); se comprueba en los finalistas.

## Etapa 2: profundidad del fine-tuning por familia

Media de los backbones de cada familia por nivel: `frozen` (0), `low` (1 stage; 6 bloques en los ViT), `high`
(2 stages; 10 bloques en los ViT) y `all`. `trainable_pct` es el % del backbone que se entrena; `vs_all` es el
cambio de PMAE de validación frente a entrenarlo todo, calculado backbone a backbone (negativo = mejor que `all`).

In [ ]:
stage = stage2_runs(load_runs(split="full"), heads)
depth_steps(stage)

## Finalistas

Nivel: `all` salvo que un nivel menor sea mejor por más del umbral de empate; la única excepción es
`dinov2_s`, que con 6 bloques (`low`) es 2.1 puntos mejor que con `all`. Cada fila es la media de las semillas
terminadas (`seeds`; `val_sd` es la SD entre ellas). Los detalles de cada familia están en
`notebooks/<familia>/results.ipynb`.

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

from nutrition5k.results import (
    backbone_costs, bar_targets, finalists, load_runs, loss_curves, overfitting_table, pareto, recipe_changes,
    target_table,
)

SPLIT = "full"  # "full" o "pilot"
RECIPE = "single-h512-224-d4-cosine"
FINALISTS = [
    f"v2-ft_all-{RECIPE}-{SPLIT}",
    f"v4m-ft_all-{RECIPE}-{SPLIT}",
    f"v4hm-ft_all-{RECIPE}-{SPLIT}",
    # f"fastvit_sa12-ft_all-{RECIPE}-{SPLIT}",  # opcional
    f"deit_t-ft_all-single-h512-224-cls+mean-d4-cosine-{SPLIT}",
    f"dinov2_s-ft_6-single-h512-224-cls+mean-d4-cosine-{SPLIT}",
    f"swin_t-ft_all-{RECIPE}-{SPLIT}",
]

runs = load_runs(split=SPLIT)
final = finalists(runs, FINALISTS)
final[["family", "backbone", "level", "seeds", "val_mean_pmae", "val_sd", "test_mean_pmae", "test_clean_mean_pmae",
       "params_M"]].round(2)

### Ruido entre semillas y umbral de empate

`val_sd` medio de los finalistas con 3 semillas. La diferencia entre las medias de dos modelos (3 semillas cada uno)
tiene una SD de `sd · √(2/3)`; el umbral es el doble de esa SD (≈ 95 % si el ruido es normal). Adoptado el 2026-11-01: 0.73
puntos. Sustituye al "~1 punto" de las etapas 1 y 2.

In [ ]:
seeded = final[final["seeds"] == 3]
sd = (seeded["val_sd"] ** 2).mean() ** 0.5
print(f"{len(seeded)} finalists with 3 seeds; pooled seed SD {sd:.2f}; threshold {2 * sd * (2 / 3) ** 0.5:.2f} points")

### `deit_t`: `all` frente a 10 bloques

Con la semilla 0, 10 bloques quedó 0.80 mejor que `all`. Con las 3 semillas la diferencia es 0.26, por debajo del umbral:
`deit_t` se queda en `all`.

In [ ]:
finalists(load_runs(["plain_vit"], SPLIT), [
    f"deit_t-ft_all-single-h512-224-cls+mean-d4-cosine-{SPLIT}",
    f"deit_t-ft_10-single-h512-224-cls+mean-d4-cosine-{SPLIT}",
])[["backbone", "level", "seeds", "val_mean_pmae", "val_sd", "test_mean_pmae"]].round(2)

### Overfitting y curvas de train y validación

Mismas columnas que en los notebooks de familia (medias entre semillas); las curvas son de la semilla 0
(discontinua: train).

In [ ]:
display(overfitting_table(final))
loss_curves(final, "backbone", "Finalists: train vs validation loss")

### Error por objetivo (validación)

In [ ]:
display(target_table(final, "backbone"))
bar_targets(final, "backbone", "Finalists: validation PMAE per target")

### Coste y Pareto (Q7)

Backbone sin la cabeza (la última fila es la cabeza); `train_*` es el nivel elegido de cada finalista. La latencia es
la de la CPU de esta máquina; la que cuenta es la del iPhone.

In [ ]:
costs = backbone_costs(final, level=None)
costs.style.format(precision=2, na_rep="—")

In [ ]:
display(pareto(final, costs, "params_M", "Finalists: validation PMAE vs backbone parameters (M)"))
pareto(final, costs, "MMACs", "Finalists: validation PMAE vs backbone MACs (millions)")

## Cambios de receta (semilla 0)

Cada run cambia una sola cosa de la receta común de un finalista (`notebooks/finalists/experiments.ipynb`,
paso 2): lr del backbone 3e-5 o 3e-4, cabeza lineal (`hidden 0 · head_width 0`) o `1024x1024 · single 256`,
weight decay 0.05, drop path 0.1, y 256 px con pesos preentrenados a 256 (`v4m`, `v4hm`). `delta` es su PMAE de
validación menos la media de 3 semillas del finalista con la receta (negativo = mejor).

Un run frente a una media de 3 semillas: la diferencia tiene una SD de `sd · √(4/3)`, así que el umbral de
criba es `2 · sd · √(4/3)` (misma regla que el de 0.73). En negrita, los cambios que lo superan: esos se
repiten con las semillas 1 y 2 y se comparan con la receta con el umbral de 0.73.


In [ ]:
changes = recipe_changes(runs, final)
screen = 2 * sd * (4 / 3) ** 0.5
print(f"{len(changes)} runs; screening threshold {screen:.2f} points (bold: better by more)")
changes.pivot(index="change", columns="backbone", values="delta").style.format("{:+.2f}", na_rep="—").map(
    lambda delta: "font-weight: bold" if delta < -screen else "")


Detalle de cada run: `val_mean_pmae_recipe` es la media de la receta, y `gap` y `late_rise` las mismas columnas
de overfitting que arriba (la regularización debería bajarlas).


In [ ]:
changes.drop(columns="path").round(2)


### Análisis (runs del paso 2 en `full`, semilla 0, 2026-11-02)

- **256 px con pesos r256:** `v4m-r256` es el único cambio que pasa la criba (−1.19: 23.36 frente a 24.55). `v4hm-r256`
  queda en −0.55. Siguiente: semillas 1 y 2 de `v4m-r256`.
- **lr del backbone:** 3e-5 es peor en los 6 (+0.4 a +1.6). 3e-4 diverge en `v4hm` (pico de pérdida en la época 5 y
  después una predicción casi constante, val ~66) y es mucho peor en `v4m` (+5.2) y `dinov2_s` (+3.3); en `v2` y
  `swin_t` mejora 0.9, sin pasar la criba. Se queda 1e-4.
- **Cabeza:** la lineal sin trunk empata con la receta en 5 de 6 (−0.90 a +0.48; +1.06 en `v4m`), y `1024x1024` no
  aporta. Con fine-tuning la capa oculta casi deja de importar (con el backbone congelado daba de 1 a 5 puntos).
- **Regularización:** ni weight decay 0.05 ni drop path 0.1 pasan la criba (lo mejor, −0.78 y −0.77), y el gap entre
  train y validación casi no cambia: el overfitting de la receta ya era pequeño.
